In [ ]:
# =============================================================================
# FINAL, CONSOLIDATED SCRIPT — with fold-wise saving + Median(IQR) + checkpoints
# (grids unchanged; CV unchanged; only adds saving/robust summaries)
# =============================================================================
import numpy as np, pandas as pd, time, tempfile, os, pickle
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold, GridSearchCV, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
import xgboost as xgb
import lightgbm as lgb
from sklearn.svm import SVR

# ---------------- 0) PATHS ----------------
DATA_PATH = r"C:\Users\kagan\Downloads\COMPO_ONLY_model_ready_with_dSref_n0913.csv"  # <- senin dosyan
OUT_DIR   = r"C:\Users\kagan\Downloads"
os.makedirs(OUT_DIR, exist_ok=True)

# checkpoint files
CKPT_PKL  = os.path.join(OUT_DIR, "final_model_results_with_folds.pkl")
CKPT_CSV  = os.path.join(OUT_DIR, "final_model_summary_with_medianIQR.csv")

# ---------------- 1) LOAD DATA ----------------
df = pd.read_csv(DATA_PATH)

# target + features
y = df["dS_ref"].astype(np.float32)
X = df.drop(columns=["dS_ref"]).astype(np.float32)

print("Dataset loaded:", X.shape, "target:", y.shape)
print("-"*60)

# ---------------- 2) QUANTILE-STRATIFIED CV (UNCHANGED) ----------------
bins = pd.qcut(y, q=5, labels=False, duplicates="drop")
cv_for_grid = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_for_eval = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
cv_list_grid = list(cv_for_grid.split(X, bins))
cv_list_eval = list(cv_for_eval.split(X, bins))

# ---------------- 3) PIPELINES (UNCHANGED) ----------------
cache_dir = tempfile.mkdtemp(prefix="sk_cache_")

pipe_rfr = Pipeline([("regressor", RandomForestRegressor(random_state=42, n_jobs=-1))],
                    memory=cache_dir)

gbr_estimator = GradientBoostingRegressor(
    loss="huber", validation_fraction=0.1, n_iter_no_change=10, tol=1e-4, random_state=42
)
pipe_gbr = Pipeline([("regressor", gbr_estimator)], memory=cache_dir)

pipe_xgb = Pipeline([("regressor", xgb.XGBRegressor(objective="reg:squarederror",
                                                    random_state=42, n_jobs=-1))],
                    memory=cache_dir)

pipe_lgbm = Pipeline([("regressor", lgb.LGBMRegressor(random_state=42, n_jobs=-1))],
                     memory=cache_dir)

pipe_svr = Pipeline([("scaler", StandardScaler()),
                     ("regressor", SVR(kernel="rbf"))],
                    memory=cache_dir)

# ---------------- 4) GRIDS (UNCHANGED) ----------------
param_grid_rfr = {
    "regressor__n_estimators": [100, 200, 400, 800],
    "regressor__max_depth": [10, 20, 30, None],
    "regressor__min_samples_split": [2, 5, 10],
    "regressor__min_samples_leaf": [1, 2, 5],
    "regressor__max_features": ["sqrt", "log2"]
}
param_grid_gbr = {
    "regressor__n_estimators": [100, 200, 400, 600, 800, 1000],
    "regressor__learning_rate": [0.01, 0.05, 0.1],
    "regressor__max_depth": [3, 5, 7, 9],
    "regressor__subsample": [0.6, 1.0],
    "regressor__max_features": ["sqrt", "log2"]
}
param_grid_xgb = {
    "regressor__n_estimators": [200, 400, 600, 800],
    "regressor__max_depth": [3, 5, 7, 9],
    "regressor__learning_rate": [0.01, 0.05, 0.1],
    "regressor__subsample": [0.6, 1.0],
    "regressor__colsample_bytree": [0.6, 1.0]
}
param_grid_lgbm_extended = {
    "regressor__n_estimators": [200, 400, 600, 800],
    "regressor__learning_rate": [0.01, 0.05, 0.1],
    "regressor__num_leaves": [20, 31, 40],
    "regressor__max_depth": [5, 10, -1],
    "regressor__subsample": [0.8, 1.0],
    "regressor__colsample_bytree": [0.8, 1.0]
}
param_grid_svr_extended = {
    "regressor__C": [1, 10, 100, 500],
    "regressor__gamma": [0.01, 0.1, "scale"]
}

models_to_run = {
    "Random Forest": (pipe_rfr, param_grid_rfr),
    "Gradient Boosting": (pipe_gbr, param_grid_gbr),
    "XGBoost": (pipe_xgb, param_grid_xgb),
    "LightGBM": (pipe_lgbm, param_grid_lgbm_extended),
    "SVR": (pipe_svr, param_grid_svr_extended)
}

# ---------------- 5) HELPERS ----------------
def median_q1_q3(arr):
    arr = np.asarray(arr, float)
    med = float(np.median(arr))
    q1, q3 = np.percentile(arr, [25, 75])
    return med, float(q1), float(q3), float(q3 - q1)

def save_checkpoint(results_dict):
    with open(CKPT_PKL, "wb") as f:
        pickle.dump(results_dict, f)
    # also write a tidy summary csv
    rows = []
    for name, d in results_dict.items():
        rows.append({
            "model": name,
            "mean_r2": d["mean_r2"], "sd_r2": d["sd_r2"],
            "median_r2": d["median_r2"], "q1_r2": d["q1_r2"], "q3_r2": d["q3_r2"], "iqr_r2": d["iqr_r2"],
            "mean_rmse": d["mean_rmse"], "sd_rmse": d["sd_rmse"],
            "median_rmse": d["median_rmse"], "q1_rmse": d["q1_rmse"], "q3_rmse": d["q3_rmse"], "iqr_rmse": d["iqr_rmse"],
            "mean_mae": d["mean_mae"], "sd_mae": d["sd_mae"],
            "median_mae": d["median_mae"], "q1_mae": d["q1_mae"], "q3_mae": d["q3_mae"], "iqr_mae": d["iqr_mae"],
            "time_taken_s": d["time_taken"]
        })
    pd.DataFrame(rows).to_csv(CKPT_CSV, index=False)

# ---------------- 6) RUN (resume-safe) ----------------
all_results = {}
if os.path.exists(CKPT_PKL):
    try:
        with open(CKPT_PKL, "rb") as f:
            all_results = pickle.load(f)
        print(f"[INFO] Loaded checkpoint: {CKPT_PKL} (models already done: {list(all_results.keys())})")
    except Exception:
        all_results = {}

scoring_metrics = ["r2", "neg_mean_squared_error", "neg_mean_absolute_error"]

for model_name, (pipeline, params) in models_to_run.items():
    if model_name in all_results:
        print(f"[SKIP] {model_name} already in checkpoint.")
        continue

    t0 = time.time()
    print(f"\n--- GridSearchCV for {model_name} ---")
    grid_search = GridSearchCV(
        estimator=pipeline,
        param_grid=params,
        scoring="r2",
        cv=cv_list_grid,
        n_jobs=-1,
        verbose=1
    )
    grid_search.fit(X, y)
    best_model = grid_search.best_estimator_
    best_params = grid_search.best_params_
    print(f"Best params for {model_name}: {best_params}")

    print(f"--- cross_validate (outer) for {model_name} ---")
    cv_scores = cross_validate(
        best_model, X, y,
        scoring=scoring_metrics,
        cv=cv_list_eval,
        n_jobs=-1,
        return_train_score=False
    )
    t1 = time.time()

    # fold-wise arrays
    fold_r2 = cv_scores["test_r2"]
    fold_rmse = np.sqrt(-cv_scores["test_neg_mean_squared_error"])
    fold_mae = -cv_scores["test_neg_mean_absolute_error"]  # make positive

    # mean±SD (use ddof=1 for sample SD)
    mean_r2 = float(np.mean(fold_r2)); sd_r2 = float(np.std(fold_r2, ddof=1))
    mean_rmse = float(np.mean(fold_rmse)); sd_rmse = float(np.std(fold_rmse, ddof=1))
    mean_mae = float(np.mean(fold_mae)); sd_mae = float(np.std(fold_mae, ddof=1))

    # median [Q1,Q3]
    med_r2, q1_r2, q3_r2, iqr_r2 = median_q1_q3(fold_r2)
    med_rmse, q1_rmse, q3_rmse, iqr_rmse = median_q1_q3(fold_rmse)
    med_mae, q1_mae, q3_mae, iqr_mae = median_q1_q3(fold_mae)

    all_results[model_name] = {
        "mean_r2": mean_r2, "sd_r2": sd_r2,
        "median_r2": med_r2, "q1_r2": q1_r2, "q3_r2": q3_r2, "iqr_r2": iqr_r2,
        "mean_rmse": mean_rmse, "sd_rmse": sd_rmse,
        "median_rmse": med_rmse, "q1_rmse": q1_rmse, "q3_rmse": q3_rmse, "iqr_rmse": iqr_rmse,
        "mean_mae": mean_mae, "sd_mae": sd_mae,
        "median_mae": med_mae, "q1_mae": q1_mae, "q3_mae": q3_mae, "iqr_mae": iqr_mae,
        "fold_r2": fold_r2, "fold_rmse": fold_rmse, "fold_mae": fold_mae,
        "best_params": best_params,
        "best_estimator": best_model,
        "time_taken": float(t1 - t0)
    }
    print(f"Done {model_name} in {t1-t0:.1f}s | R2 mean±SD={mean_r2:.3f}±{sd_r2:.3f} | median[Q1,Q3]={med_r2:.3f}[{q1_r2:.3f},{q3_r2:.3f}]")

    # checkpoint after each model
    save_checkpoint(all_results)
    print(f"[INFO] Checkpoint saved -> {CKPT_PKL}")
    print(f"[INFO] Summary CSV saved -> {CKPT_CSV}")

# ---------------- 7) FINAL REPORT ----------------
final_df = pd.DataFrame({
    m: {
        "R2 (Mean±SD)": f'{d["mean_r2"]:.3f} ± {d["sd_r2"]:.3f}',
        "RMSE (Mean±SD)": f'{d["mean_rmse"]:.3f} ± {d["sd_rmse"]:.3f}',
        "MAE (Mean±SD)": f'{d["mean_mae"]:.3f} ± {d["sd_mae"]:.3f}',
        "R2 (Median [Q1,Q3])": f'{d["median_r2"]:.3f} [{d["q1_r2"]:.3f}, {d["q3_r2"]:.3f}]',
        "RMSE (Median [Q1,Q3])": f'{d["median_rmse"]:.3f} [{d["q1_rmse"]:.3f}, {d["q3_rmse"]:.3f}]',
        "MAE (Median [Q1,Q3])": f'{d["median_mae"]:.3f} [{d["q1_mae"]:.3f}, {d["q3_mae"]:.3f}]'
    } for m, d in all_results.items()
}).T

final_df = final_df.sort_values(by="R2 (Mean±SD)", ascending=False)

print("\n" + "="*70)
print("FINAL MODEL COMPARISON (Mean±SD and Median [Q1,Q3])")
print("="*70)
print(final_df)
print("\nSaved:", CKPT_PKL)
print("Saved:", CKPT_CSV)